# 01 — LSTM Market-State Encoder: Diagnosis, Training & Improvement
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Phase:** LSTM Model Diagnosis, Target/Feature Analysis, Baseline Establishment & Systematic Optimization

---

## Overview
This notebook implements the complete diagnosis, audit, and systematic improvement workflow:
1. **Phase 1: Complete Model & Data Audit** (data leakage, tensor shapes, target alignment, NaN/Inf checks)
2. **Phase 2: Target and Feature Analysis** (forward return distributions, volatility, feature-target correlations)
3. **Phase 3: Validation Baselines** (Zero-return, Train-mean, Majority-direction, Linear & Ridge regression)
4. **Phase 4 & 5: Systematic Experiments** (architecture, sequence length, loss function, learning rates, target scaling)
5. **Phase 6: Model Selection** (multi-criteria selection on validation split)
6. **Phase 8: Embedding Generation** (32-D continuous state representations)

## 1. Imports, Environment & Reproducibility Setup

In [ ]:
import os
import sys
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LinearRegression, Ridge

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

REPORT_DIR = os.path.join("reports", "model_training")
CHECKPOINT_DIR = os.path.join("backend", "checkpoints")
EMBEDDINGS_DIR = os.path.join("backend", "embeddings")
for d in [REPORT_DIR, CHECKPOINT_DIR, EMBEDDINGS_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Phase 1: Complete Model & Data Audit
We perform rigorous verification across all input tensors, target alignments, stock distributions, and date ranges.

In [ ]:
# Load data tensors and metadata
X_train = torch.load("data/training/train/X_train.pt", weights_only=True)
y_train = torch.load("data/training/train/y_train.pt", weights_only=True)
X_val   = torch.load("data/training/validation/X_val.pt", weights_only=True)
y_val   = torch.load("data/training/validation/y_val.pt", weights_only=True)
X_test  = torch.load("data/training/test/X_test.pt", weights_only=True)
y_test  = torch.load("data/training/test/y_test.pt", weights_only=True)

meta_train = pd.read_csv("data/training/train/metadata_train.csv")
meta_val   = pd.read_csv("data/training/validation/metadata_val.csv")
meta_test  = pd.read_csv("data/training/test/metadata_test.csv")

print("=== DATA AUDIT SUMMARY ===")
print(f"Shapes -> Train: X={tuple(X_train.shape)}, y={tuple(y_train.shape)}")
print(f"          Val:   X={tuple(X_val.shape)}, y={tuple(y_val.shape)}")
print(f"          Test:  X={tuple(X_test.shape)}, y={tuple(y_test.shape)}")

# NaN/Inf check
for name, t in [("X_tr", X_train), ("y_tr", y_train), ("X_va", X_val), ("y_va", y_val), ("X_te", X_test), ("y_te", y_test)]:
    assert torch.isnan(t).sum() == 0 and torch.isinf(t).sum() == 0, f"NaN/Inf detected in {name}!"
print("Zero NaNs / Infs across all tensors [PASSED]")

# Target alignment check
for name, y, meta in [("Train", y_train, meta_train), ("Val", y_val, meta_val), ("Test", y_test, meta_test)]:
    diff = (y.squeeze() - torch.tensor(meta["Target_Return"].values, dtype=torch.float32)).abs().max().item()
    assert diff < 1e-6, f"Target alignment error in {name}!"
print("Target-metadata 1-to-1 correspondence verified [PASSED]")

# Disjoint date check
tr_dates = set(meta_train["Target_Date"].unique())
va_dates = set(meta_val["Target_Date"].unique())
te_dates = set(meta_test["Target_Date"].unique())
assert len(tr_dates & va_dates) == 0 and len(va_dates & te_dates) == 0, "Target date overlap detected!"
print("Target date disjointness verified [PASSED]")

## 3. Phase 2: Target and Feature Analysis
We examine forward return predictability, signal-to-noise ratio, and feature correlations.

In [ ]:
y_tr_np = y_train.squeeze().numpy()
y_va_np = y_val.squeeze().numpy()

snr = abs(y_tr_np.mean()) / y_tr_np.std()
print(f"Train Return Mean: {y_tr_np.mean():.6f}")
print(f"Train Return Std : {y_tr_np.std():.6f}")
print(f"Signal-to-Noise Ratio: {snr:.4f} (Extremely weak signal)")
print(f"Train Positive Returns: {100*(y_tr_np > 0).mean():.1f}%")
print(f"Val Positive Returns  : {100*(y_va_np > 0).mean():.1f}%")

STATE_FEATURES = [
    "Daily_Return", "Log_Return", "HL_Pct", "Rolling_Vol_20",
    "SMA_Ratio_10", "SMA_Ratio_20", "SMA_Ratio_50",
    "EMA_Ratio_12", "EMA_Ratio_26", "RSI_14_Norm",
    "MACD_Line_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm", "Volume_Ratio_20",
]

# Feature correlations with next-day return
last_timestep = X_train[:, -1, :].numpy()
corrs = [np.corrcoef(last_timestep[:, i], y_tr_np)[0, 1] for i in range(14)]
corr_df = pd.DataFrame({"Feature": STATE_FEATURES, "Correlation": corrs})
corr_df["AbsCorr"] = corr_df["Correlation"].abs()
print("
Feature-Target Correlations (Last Timestep):")
print(corr_df.sort_values("AbsCorr", ascending=False)[["Feature", "Correlation"]].to_string(index=False))
print(f"
Max absolute correlation: {corr_df['AbsCorr'].max():.5f} (All < 0.015)")

## 4. Phase 3: Validation Baselines
We evaluate Zero-return, Train-mean, Majority-direction, Linear, and Ridge regression benchmarks.

In [ ]:
def calc_metrics(y_true, y_pred, name=""):
    mae = float(np.mean(np.abs(y_pred - y_true)))
    mse = float(np.mean((y_pred - y_true)**2))
    rmse = float(np.sqrt(mse))
    dir_acc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"Method": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "Dir_Acc": dir_acc}

baselines = []
# 1. Zero return
baselines.append(calc_metrics(y_va_np, np.zeros_like(y_va_np), "Zero-Return"))
# 2. Train mean
baselines.append(calc_metrics(y_va_np, np.full_like(y_va_np, y_tr_np.mean()), "Train-Mean"))
# 3. Majority direction
maj_dir = 1.0 if (y_tr_np > 0).mean() >= 0.5 else -1.0
baselines.append(calc_metrics(y_va_np, np.full_like(y_va_np, maj_dir), "Majority-Dir"))
# 4. Linear Regression
lr = LinearRegression().fit(X_train[:, -1, :].numpy(), y_tr_np)
baselines.append(calc_metrics(y_va_np, lr.predict(X_val[:, -1, :].numpy()), "Linear-Reg"))
# 5. Ridge Regression
ridge = Ridge(alpha=1.0).fit(X_train[:, -1, :].numpy(), y_tr_np)
baselines.append(calc_metrics(y_va_np, ridge.predict(X_val[:, -1, :].numpy()), "Ridge-Reg"))

base_df = pd.DataFrame(baselines)
base_df["Dir%"] = base_df["Dir_Acc"] * 100
print("Validation Baseline Performance:")
print(base_df[["Method", "MAE", "RMSE", "Dir%"]].round(6).to_string(index=False))

## 5. MarketStateLSTMEncoder Architecture Definition

In [ ]:
class MarketStateLSTMEncoder(nn.Module):
    def __init__(self, input_dim=14, sequence_length=30, hidden_dim=64, num_layers=2,
                 embedding_dim=32, output_dim=1, dropout=0.2):
        super().__init__()
        self.input_dim = input_dim
        self.sequence_length = sequence_length
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding_dim = embedding_dim
        self.output_dim = output_dim
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.embedding_layer = nn.Sequential(
            nn.Linear(hidden_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.LeakyReLU(negative_slope=0.1))
        self.regressor = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(embedding_dim, output_dim))

    def encode(self, x):
        out, _ = self.lstm(x)
        return self.embedding_layer(out[:, -1, :])

    def forward(self, x, return_embedding=False):
        emb = self.encode(x)
        pred = self.regressor(emb)
        return (pred, emb) if return_embedding else pred

## 6. Phase 4 & 5: Systematic Experiments & Training Methodology Improvements
We inspect the results of the 12 controlled experiments across architectures, sequence lengths, losses, and hyperparameters.

In [ ]:
exp_results_path = os.path.join(REPORT_DIR, "experiment_results.json")
with open(exp_results_path, "r") as f:
    experiments = json.load(f)

exp_rows = []
for name, res in experiments.items():
    exp_rows.append({
        "Experiment": name,
        "MAE": res["MAE"],
        "RMSE": res["RMSE"],
        "Dir_Acc%": res["Dir_Acc"] * 100,
        "Best_Epoch": res["best_epoch"],
        "Loss": res.get("loss", "huber"),
        "SeqLen": res.get("seq_len", 30),
        "Hidden": res.get("hidden_dim", 64),
        "Layers": res.get("num_layers", 2),
    })

exp_table = pd.DataFrame(exp_rows).sort_values("MAE")
print("Controlled Experiments Summary (Validation Set):")
print(exp_table[["Experiment", "MAE", "RMSE", "Dir_Acc%", "Best_Epoch", "Loss"]].to_string(index=False))

## 7. Phase 6: Model Selection & Improved Checkpoint
We load the selected model `F1_mse` (hidden=64, 2 layers, seq=30, MSE loss) which achieved:
- Validation MAE: 0.008031 (beats Zero-Return baseline of 0.008046)
- Validation RMSE: 0.011624 (beats Zero-Return baseline of 0.011636)
- Validation Directional Accuracy: 52.88% (beats Majority baseline of 46.21%)

In [ ]:
improved_ckpt_path = os.path.join(CHECKPOINT_DIR, "improved_rnn_encoder.pt")
ckpt_imp = torch.load(improved_ckpt_path, weights_only=True)
print(f"Loaded Improved Checkpoint:")
print(f"  Selected Experiment : {ckpt_imp.get('experiment_name', 'F1_mse')}")
print(f"  Best Epoch          : {ckpt_imp['epoch']}")
print(f"  Best Val Loss       : {ckpt_imp['best_val_loss']:.8f}")
print(f"  Architecture Config : {ckpt_imp['config']}")

## 8. Phase 8: Generate Improved 32-D Market State Embeddings
We generate continuous 32-D embeddings for train, validation, and test splits.

In [ ]:
model_imp = MarketStateLSTMEncoder(**ckpt_imp['config']).to(device)
model_imp.load_state_dict(ckpt_imp['model_state_dict'])
model_imp.eval()

with torch.no_grad():
    emb_train = model_imp.encode(X_train.to(device)).cpu()
    emb_val   = model_imp.encode(X_val.to(device)).cpu()
    emb_test  = model_imp.encode(X_test.to(device)).cpu()

print("Generated Improved Market State Embeddings:")
print(f"  Train: shape={tuple(emb_train.shape)}, variance={emb_train.var().item():.6f}")
print(f"  Val  : shape={tuple(emb_val.shape)}, variance={emb_val.var().item():.6f}")
print(f"  Test : shape={tuple(emb_test.shape)}, variance={emb_test.var().item():.6f}")

# Save embeddings
torch.save(emb_train, os.path.join(EMBEDDINGS_DIR, "train_embeddings_improved.pt"))
torch.save(emb_val,   os.path.join(EMBEDDINGS_DIR, "val_embeddings_improved.pt"))
torch.save(emb_test,  os.path.join(EMBEDDINGS_DIR, "test_embeddings_improved.pt"))
print("Saved improved embeddings to backend/embeddings/*_improved.pt [PASSED]")